# Part 3: Filtering and the marginal log-likelihood

We use **filtering** (e.g. the cuthbert particle filter) to compute the **marginal log-likelihood** (MLL) $\log p(y_{1:T} \mid \theta)$ at fixed parameters $\theta$, and show how to switch to a different filter (e.g. Taylor KF).

## 3.1 Why filtering? Computing MLL without sampling states

Sometimes we don't want to sample the full state trajectory—we only care about **parameters** $\theta$. The **marginal log-likelihood** is $\log p(y_{1:T} \mid \theta)$ with states integrated out. Dynestyx provides **filtering** handlers that add this MLL as a NumPyro factor, so we can evaluate $\log p(\text{data} \mid \theta)$ at any $\theta$.

We'll use the **particle filter** (cuthbert PF) to compute the MLL at:

1. **Posterior mean** $\theta_{\text{mean}}$ (e.g. from the NUTS posterior from [Part 2](../02_dynestyx_discrete_intro.ipynb)),
2. **True** $\theta_{\text{true}}$ (the generating parameters, for comparison).

We evaluate the model at fixed parameters (as when using `Predictive` with `params=...`) and read the MLL factor from the trace.

In [1]:
import jax.numpy as jnp
import jax.random as jr

from dynestyx import Context, Condition, Trajectory, FilterBasedMarginalLogLikelihood, DynamicalModel, DiscreteTimeSimulator

# Standalone: define the model and generate data (same as Part 2)
import numpyro
import numpyro.distributions as dist
import dynestyx as dsx
from numpyro.infer import Predictive, MCMC, NUTS

def stochastic_volatility_model():
    phi = numpyro.sample("phi", dist.Uniform(0.0, 1.0))
    sigma_eta = 0.5
    initial_condition = dist.Normal(0.0, 1.0)
    def state_evolution(x, u, t_now, t_next):
        return dist.Normal(phi * x, sigma_eta)
    def observation_model(x, u, t):
        return dist.Normal(0.0, jnp.exp(x / 2.0))
    dynamics = DynamicalModel(
        state_dim=1, observation_dim=1, control_dim=0,
        initial_condition=initial_condition,
        state_evolution=state_evolution,
        observation_model=observation_model,
    )
    return dsx.sample("f", dynamics)

key = jr.PRNGKey(0)
phi_true = 0.9
obs_times = jnp.arange(0.0, 100.0, 1.0)
context_gen = Context(observations=Trajectory(times=obs_times), controls=Trajectory())
predictive = Predictive(stochastic_volatility_model, params={"phi": jnp.array(phi_true)}, num_samples=1, exclude_deterministic=False)
with DiscreteTimeSimulator():
    with Condition(context_gen):
        synthetic = predictive(key)
obs_values = synthetic["observations"][0]
observation_trajectory = Trajectory(times=obs_times, values=obs_values)

In [2]:
def filter_conditioned_model():
    """Model that uses PF to add MLL factor; no state sampling."""
    context = Context(observations=observation_trajectory, controls=Trajectory())
    with FilterBasedMarginalLogLikelihood(filter_type="pf", n_filter_particles=2000):
        with Condition(context):
            return stochastic_volatility_model()

# Run NUTS to get posterior (reuse from Part 2 logic)
def data_conditioned_model():
    context = Context(observations=observation_trajectory, controls=Trajectory())
    with DiscreteTimeSimulator():
        with Condition(context):
            return stochastic_volatility_model()
nuts_kernel = NUTS(data_conditioned_model)
mcmc = MCMC(nuts_kernel, num_warmup=200, num_samples=200)
mcmc.run(jr.PRNGKey(1))
posterior_sv = mcmc.get_samples()

theta_mean = {k: jnp.mean(v, axis=0) for k, v in posterior_sv.items()}
theta_true = {"phi": jnp.array(phi_true)}

sample: 100%|██████████| 400/400 [00:00<00:00, 782.38it/s, 15 steps of size 2.42e-01. acc. prob=0.87]


In [3]:
posterior_sv.keys()

dict_keys(['observations', 'phi', 'states', 'times', 'x_0', 'x_Traced<int32[]>with<DynamicJaxprTrace>'])

In [6]:
def get_mll(model_fn, params, key=jr.PRNGKey(0)):
    """Evaluate model at fixed params via Predictive and return the MLL (deterministic site)."""
    predictive = Predictive(
        model_fn, params=params, num_samples=1, exclude_deterministic=False
    )
    result = predictive(key)
    for name, value in result.items():
        if name.endswith("_marginal_loglik"):
            return float(value[0])
    return None

print("MLL at theta_mean (posterior mean):", get_mll(filter_conditioned_model, theta_mean))
print("MLL at theta_true (true generating params):", get_mll(filter_conditioned_model, theta_true))

MLL at theta_mean (posterior mean): -134.95135498046875
MLL at theta_true (true generating params): -134.52108764648438


The MLL at $\theta_{\text{true}}$ is the log-likelihood of the data under the true model; the MLL at $\theta_{\text{mean}}$ tells us how well the posterior mean explains the data. Typically the true parameters achieve a higher (or similar) MLL.

## 3.2 Switching to a different filter: Taylor KF

For discrete-time models, dynestyx can use different filters via the **cuthbert** backend:

- **`filter_type="pf"`**: particle filter (works for general observation models; we used it above).
- **`filter_type="taylor_kf"`**: Taylor-linearized Kalman filter (faster, but assumes a linear-Gaussian approximation; good when the observation model is close to linear-Gaussian).

You only change the handler: use `FilterBasedMarginalLogLikelihood(filter_type="taylor_kf")` instead of `"pf"`. The stochastic volatility observation model is non-linear in $x_t$, so the Taylor KF gives an approximation; for linear Gaussian models it yields the exact MLL.

**Canonical reference for particle filtering and pseudomarginal MCMC**: *[CITATION PLACEHOLDER — to be added]*

In [7]:
def filter_conditioned_model_taylor_kf():
    context = Context(observations=observation_trajectory, controls=Trajectory())
    with FilterBasedMarginalLogLikelihood(filter_type="taylor_kf"):
        with Condition(context):
            return stochastic_volatility_model()

print("MLL at theta_true (Taylor KF approx):", get_mll(filter_conditioned_model_taylor_kf, theta_true))

ValueError: diag input must be 1d or 2d

**Next:** [Part 4 — Filtering + NUTS: pseudomarginal inference](../04_filtering_nuts_pseudomarginal.ipynb)